# 15. Interval re-report of every threshold-dependent table

Notebook 14 showed that a 1 percent FPR threshold on 399 calibration benigns is fixed by the top four benign scores, and that the published thresholds sit in data-free gaps between them. So every FNR in the paper is only identified up to an interval. This notebook re-reports each threshold-dependent table with three values per cell:

- **lo**: threshold at the `lower` benign order statistic (same calibration FPR as published, low end of the gap)
- **hi**: threshold at the `higher` benign order statistic (same calibration FPR as published, high end of the gap)
- **guar**: the smallest benign score that guarantees calibration FPR at or below 1 percent

Threshold-free metrics (AUROC, ECE_atk, Brier_atk) are reported once and checked to be unchanged. Cached scores only, no GPU. Stages that need the phase1, phase3 or phase5 caches skip with a message if the files are absent.

**Outputs**: `reports/panel_interval.csv` (Table 2), `reports/cci_interval.csv` (Table 4), `reports/source_threshold_interval.csv` (Table 9), `reports/exploitable_miss_interval.csv` (Table 5), `reports/adversary_interval.csv` (Section 4.11), `reports/judge_panel_interval.csv` (Table 7), `figures/panel_interval.png`.

In [ ]:
# === SESSION BOOTSTRAP (run first) ===
from google.colab import drive
drive.mount('/content/drive')
import os, sys
DRIVE_ROOT = '/content/drive/MyDrive/PICALIB_Research'
REPO_DIR   = os.path.join(DRIVE_ROOT, 'picalib-research')
!git config --global user.name  "Md Anas Biswas"
!git config --global user.email "anasbiswas@gmail.com"
!git config --global credential.helper "store --file={DRIVE_ROOT}/.git-credentials"
%cd {REPO_DIR}
sys.path.insert(0, 'src')
!git pull
os.makedirs('tools', exist_ok=True); os.makedirs('reports', exist_ok=True); os.makedirs('figures', exist_ok=True)
print('Session ready.')

## Stage 0. Commit tool with a secret scan

Same contract as before, plus: after staging, every staged text file is scanned for token patterns and the commit is refused, naming the file, if any are found. This is the check GitHub's push protection did for us in notebook 13; now it runs locally first.

In [ ]:
%%writefile tools/commit_cell.py
"""Single commit step for this repo. Run from a notebook's final cell only:

    subprocess.run(["python", "tools/commit_cell.py", "message"])

Restores git identity and credentials from the Drive project folder, strips all notebook outputs,
refuses to stage data, credential or oversized files, refuses any staged file containing a secret
token pattern, commits if anything is staged, and always pushes so unpushed commits never remain local.
"""
from __future__ import annotations

import os
import re
import shutil
import subprocess
import sys
from glob import glob
from pathlib import Path

DRIVE_PROJECT_ROOT = Path("/content/drive/MyDrive/PICALIB_Research")
REPO_ROOT = Path(os.environ.get("PICALIB_REPO_ROOT", str(DRIVE_PROJECT_ROOT / "picalib-research")))
GIT_NAME = "Md Anas Biswas"
GIT_EMAIL = "anasbiswas@gmail.com"
FORBIDDEN_NAMES = {".git-credentials", ".gitconfig", "kaggle.json", ".env"}
FORBIDDEN_PREFIXES = ("data/", "_bipia/", ".secrets/")
MAX_FILE_MB = 50
SECRET_PATTERNS = [
    re.compile(r"hf_[A-Za-z0-9]{20,}"),            # Hugging Face
    re.compile(r"ghp_[A-Za-z0-9]{30,}"),           # GitHub classic PAT
    re.compile(r"github_pat_[A-Za-z0-9_]{40,}"),   # GitHub fine-grained PAT
    re.compile(r"sk-[A-Za-z0-9]{32,}"),             # OpenAI-style keys
    re.compile(r"AKIA[0-9A-Z]{16}"),               # AWS access key id
    re.compile(r"AIza[0-9A-Za-z_\-]{35}"),         # Google API key
]
TEXT_EXT = {".py", ".ipynb", ".md", ".txt", ".json", ".csv", ".yaml", ".yml", ".toml", ".cfg", ".ini", ".sh"}


def run(args, check=True):
    r = subprocess.run(args, capture_output=True, text=True)
    if check and r.returncode != 0:
        raise SystemExit(f"{' '.join(args)}\n{r.stdout}{r.stderr}")
    return r.stdout


def strip_outputs():
    notebooks = sorted(glob(str(REPO_ROOT / "notebooks" / "*.ipynb")))
    for notebook in notebooks:
        r = subprocess.run([sys.executable, "-m", "jupyter", "nbconvert",
                            "--ClearOutputPreprocessor.enabled=True", "--inplace", notebook],
                           capture_output=True, text=True)
        if r.returncode != 0:
            import nbformat
            nb = nbformat.read(notebook, as_version=4)
            for cell in nb.cells:
                if cell.cell_type == "code":
                    cell["outputs"] = []; cell["execution_count"] = None
            nbformat.write(nb, notebook)
    import nbformat
    for notebook in notebooks:
        nb = nbformat.read(notebook, as_version=4)
        if "widgets" in nb.metadata:
            del nb.metadata["widgets"]; nbformat.write(nb, notebook)
    print(f"outputs: stripped {len(notebooks)} notebook(s)")


def secret_scan(paths):
    hits = []
    for p in paths:
        path = REPO_ROOT / p
        if path.suffix.lower() not in TEXT_EXT or not path.is_file():
            continue
        try:
            txt = path.read_text(encoding="utf-8", errors="ignore")
        except Exception:
            continue
        for pat in SECRET_PATTERNS:
            if pat.search(txt):
                hits.append(f"{p} ({pat.pattern[:14]}...)")
                break
    return hits


def main() -> None:
    if len(sys.argv) < 2 or not sys.argv[1].strip():
        raise SystemExit('usage: python tools/commit_cell.py "commit message"')
    message = sys.argv[1].strip()

    for dotfile in (".gitconfig", ".git-credentials"):
        source = DRIVE_PROJECT_ROOT / dotfile
        if source.exists():
            shutil.copy(source, Path.home() / dotfile)
    credentials = Path.home() / ".git-credentials"
    if credentials.exists():
        os.chmod(credentials, 0o600)

    os.chdir(REPO_ROOT)
    run(["git", "config", "user.name", GIT_NAME])
    run(["git", "config", "user.email", GIT_EMAIL])
    run(["git", "config", "credential.helper", "store"])

    strip_outputs()

    run(["git", "add", "-A"])
    staged = [l for l in run(["git", "diff", "--cached", "--name-only"]).splitlines() if l.strip()]
    bad = [p for p in staged if Path(p).name in FORBIDDEN_NAMES or p.startswith(FORBIDDEN_PREFIXES)]
    big = [p for p in staged if Path(p).exists() and Path(p).stat().st_size > MAX_FILE_MB * 1024 * 1024]
    secrets = secret_scan(staged)
    if bad or big or secrets:
        run(["git", "reset", "-q"], check=False)
        raise SystemExit(f"refused: forbidden={bad} oversized={big} secrets_in={secrets}")

    if staged:
        commit = subprocess.run(["git", "commit", "-m", message], capture_output=True, text=True)
        print(commit.stdout or commit.stderr)
    else:
        print("nothing new to commit; pushing any unpushed commits")

    push = subprocess.run(["git", "push", "origin", "main"], capture_output=True, text=True)
    print(push.stdout or push.stderr)
    print(run(["git", "log", "--oneline", "-1"]).strip())
    print(run(["git", "status", "--short"]).strip() or "working tree clean")


if __name__ == "__main__":
    main()


## Stage A. Load frames and cached frozen scores (as notebooks 13 and 14)

In [ ]:
import importlib, numpy as np, pandas as pd, os, json, glob
import data_loaders, detectors, metrics
for m in (data_loaders, detectors, metrics): importlib.reload(m)
from data_loaders import (load_deepset, load_bipia_categorized, load_notinject, load_jailbreak,
                          BIPIA_HARMFUL, BIPIA_HIJACK, BIPIA_MALICIOUS)
from detectors import threshold_at_fpr, auroc

deepset = load_deepset()
bip     = load_bipia_categorized(per_category=30, categories=BIPIA_MALICIOUS)
noti    = load_notinject()
jb      = load_jailbreak()
FRAMES  = {'deepset': deepset, 'bipia_cat': bip, 'notinject': noti, 'jailbreak': jb}

TAGS = ['protectai_v2', 'prompt_guard_2', 'prompt_guard_2_22m']
LABEL = {'protectai_v2': 'ProtectAI-v2', 'prompt_guard_2': 'Prompt-Guard-2 (86M)', 'prompt_guard_2_22m': 'Prompt-Guard-2 (22M)'}
P, missing = {}, []
for tag in TAGS:
    P[tag] = {}
    for key, f in FRAMES.items():
        fp = f'data/score_{tag}_{key}.npy'
        if not os.path.exists(fp): missing.append(fp); continue
        p = np.load(fp); assert len(p) == len(f), f'{fp}: {len(p)} vs {len(f)} rows'
        P[tag][key] = p
if missing: raise SystemExit('Missing score caches:\n  ' + '\n  '.join(missing))

yd = deepset.label.values
bipben = bip.label.values == 0
H = bip.meta.isin(BIPIA_HARMFUL).values
J = bip.meta.isin(BIPIA_HIJACK).values
def cells_for(tag):
    p = P[tag]; cells = {}
    cells['direct']           = (yd, p['deepset'])
    cells['indirect_harmful'] = (np.r_[np.ones(H.sum()), np.zeros(bipben.sum())], np.r_[p['bipia_cat'][H], p['bipia_cat'][bipben]])
    cells['indirect_hijack']  = (np.r_[np.ones(J.sum()), np.zeros(bipben.sum())], np.r_[p['bipia_cat'][J], p['bipia_cat'][bipben]])
    cells['jailbreak']        = (jb.label.values, p['jailbreak'])
    cells['over_defense']     = (np.zeros(len(noti)), p['notinject'])
    return cells, p['deepset'][yd == 0]
SHIFTS = ['direct', 'indirect_harmful', 'indirect_hijack', 'jailbreak']
CELLS = {tag: cells_for(tag) for tag in TAGS}
print('loaded:', {tag: {k: int((y==1).sum()) for k,(y,p) in CELLS[tag][0].items()} for tag in TAGS})

## Stage B. Interval module and the Table 2 re-report

In [ ]:
%%writefile src/interval_panel.py
"""Interval re-reporting for frozen-threshold transport panels.

For a target FPR on a benign calibration set, the threshold is identified only up to the gap between
two consecutive benign order statistics. Every threshold-dependent metric is therefore reported at the
low end (t_lower), the high end (t_higher) and at the guaranteed-FPR threshold (t_guar).
"""
from __future__ import annotations
import numpy as np
from score_diagnostics import thresholds as _thresholds

KEYS = ("lo", "hi", "guar")


def three_thresholds(p_ben_cal, fpr=0.01):
    T = _thresholds(p_ben_cal, fpr)
    return {"lo": T["t_lower"], "hi": T["t_higher"], "guar": T["t_guar"]}


def fnr(pa, t):
    return float((pa < t).mean()) if len(pa) else np.nan


def severity(pa, t, min_misses=10):
    m = pa < t
    return float(np.mean(1 - pa[m])) if m.sum() >= min_misses else np.nan


def fpr(pb, t):
    return float((pb >= t).mean()) if len(pb) else np.nan


def bootstrap_fnr(pa, t, B=1000, seed=0):
    rng = np.random.default_rng(seed); n = len(pa)
    v = [fnr(pa[rng.integers(0, n, n)], t) for _ in range(B)]
    return tuple(np.percentile(v, [2.5, 97.5]))


def interval_cell(pa, pb_cal, pb_tgt=None, fpr_target=0.01, auroc_fn=None):
    """All three thresholds' worth of metrics for one detector x shift cell."""
    T = three_thresholds(pb_cal, fpr_target); out = {f"t_{k}": T[k] for k in KEYS}
    for k in KEYS:
        t = T[k]
        out[f"FNR_{k}"] = fnr(pa, t)
        out[f"FNR_{k}_lo95"], out[f"FNR_{k}_hi95"] = bootstrap_fnr(pa, t)
        out[f"n_miss_{k}"] = int((pa < t).sum())
        out[f"S_{k}"] = severity(pa, t)
        out[f"calFPR_{k}"] = fpr(pb_cal, t)
        if pb_tgt is not None and len(pb_tgt):
            out[f"tgtFPR_{k}"] = fpr(pb_tgt, t)
    out["FNR_width_lo_hi"] = out["FNR_hi"] - out["FNR_lo"]
    if pb_tgt is not None and len(pb_tgt) and auroc_fn is not None:
        y = np.r_[np.ones(len(pa)), np.zeros(len(pb_tgt))]
        out["AUROC"] = auroc_fn(y, np.r_[pa, pb_tgt])
    return out


def rel_change(v_shift, v_direct):
    return float((v_shift - v_direct) / v_direct) if v_direct not in (0, None) and not np.isnan(v_direct) else np.nan


In [ ]:
import importlib, score_diagnostics, interval_panel
importlib.reload(score_diagnostics); importlib.reload(interval_panel)
from interval_panel import interval_cell, three_thresholds, fnr, severity, fpr, rel_change, KEYS

rows = []
for tag in TAGS:
    cells, ben_cal = CELLS[tag]
    for sh in SHIFTS:
        y, p = cells[sh]; pa, pb = p[y == 1], p[y == 0]
        r = interval_cell(pa, ben_cal, pb_tgt=pb, auroc_fn=auroc)
        rows.append(dict(detector=LABEL[tag], shift=sh, n_atk=int(len(pa)), **r))
    # over-defense row: benign-only, report target FPR at the three thresholds
    y, p = cells['over_defense']; T = three_thresholds(ben_cal)
    rows.append(dict(detector=LABEL[tag], shift='over_defense', n_atk=0, **{f't_{k}': T[k] for k in KEYS},
                     **{f'tgtFPR_{k}': fpr(p, T[k]) for k in KEYS}))
PI = pd.DataFrame(rows)
PI.to_csv('reports/panel_interval.csv', index=False)
pd.set_option('display.width', 250)
show = ['detector','shift','n_atk','FNR_lo','FNR_hi','FNR_guar','FNR_width_lo_hi','S_lo','S_hi','S_guar','tgtFPR_lo','tgtFPR_hi','tgtFPR_guar','AUROC']
print('=== Table 2 re-reported as intervals (lo / hi share the published calibration FPR; guar <= 1 percent) ===')
print(PI[show].round(3).to_string(index=False))
print('\ncells where the lo-hi FNR interval is at least 0.10 wide:')
w = PI[PI.FNR_width_lo_hi >= 0.10][['detector','shift','FNR_lo','FNR_hi','FNR_width_lo_hi']]
print('NONE' if w.empty else w.round(3).to_string(index=False))

## Stage C. Table 3 severity column and Table 4 CCI as intervals

ECE_atk and Brier_atk are threshold-free; they are recomputed here and checked to be identical across the three thresholds. S and FNR change; the CCI (relative change from the direct baseline) is reported at each threshold.

In [ ]:
from metrics import ece_attack_conditional
rows = []
for tag in TAGS:
    cells, ben_cal = CELLS[tag]; T = three_thresholds(ben_cal)
    for sh in SHIFTS:
        y, p = cells[sh]; pa = p[y == 1]
        e = {k: float(ece_attack_conditional(y, p, t=T[k])) for k in KEYS}
        assert max(e.values()) - min(e.values()) < 1e-12, f'ECE_atk moved with threshold in {tag}/{sh}: {e}'
        rows.append(dict(detector=LABEL[tag], shift=sh, ECE_atk=e['lo'], Brier_atk=float(np.mean((1 - pa) ** 2)),
                         **{f'S_{k}': severity(pa, T[k]) for k in KEYS}))
T3 = pd.DataFrame(rows); T3.to_csv('reports/table3_interval.csv', index=False)
print('=== Table 3: threshold-free ECE_atk and Brier_atk confirmed unchanged; S at three thresholds ===')
print(T3.round(3).to_string(index=False))

# CCI: relative change from the direct cell, per threshold, for FNR, S, AUROC, ECE_atk
rows = []
for tag in TAGS:
    d = PI[(PI['detector'] == LABEL[tag])].set_index('shift')
    base = d.loc['direct']; e = T3[T3['detector'] == LABEL[tag]].set_index('shift')
    for sh in ['indirect_harmful', 'indirect_hijack']:
        r = dict(detector=LABEL[tag], shift=sh,
                 CCI_AUROC=rel_change(d.loc[sh, 'AUROC'], base['AUROC']),
                 CCI_ECE_atk=rel_change(e.loc[sh, 'ECE_atk'], e.loc['direct', 'ECE_atk']))
        for k in KEYS:
            r[f'CCI_FNR_{k}'] = rel_change(d.loc[sh, f'FNR_{k}'], base[f'FNR_{k}'])
            r[f'CCI_S_{k}'] = rel_change(d.loc[sh, f'S_{k}'], base[f'S_{k}'])
        rows.append(r)
CCI = pd.DataFrame(rows); CCI.to_csv('reports/cci_interval.csv', index=False)
print('\n=== Table 4 CCI at the three thresholds (AUROC and ECE_atk columns are threshold-free) ===')
print(CCI.round(3).to_string(index=False))

## Stage D. Table 9 source-threshold transport as intervals

Each source's benign pool gives its own three thresholds. The oracle for the indirect targets is the BIPIA host pool. Reported: FNR on each target at lo, hi and guar for every source.

In [ ]:
SOURCES = {'deepset': lambda tag: P[tag]['deepset'][yd == 0],
           'jailbreak': lambda tag: P[tag]['jailbreak'][jb.label.values == 0],
           'bipia_host': lambda tag: P[tag]['bipia_cat'][bipben],
           'notinject': lambda tag: P[tag]['notinject']}
TARGETS_T9 = {'direct': lambda tag: P[tag]['deepset'][yd == 1],
              'indirect_harmful': lambda tag: P[tag]['bipia_cat'][H],
              'indirect_hijack': lambda tag: P[tag]['bipia_cat'][J],
              'jailbreak': lambda tag: P[tag]['jailbreak'][jb.label.values == 1]}
rows = []
for tag in TAGS:
    for src, fb in SOURCES.items():
        T = three_thresholds(fb(tag))
        for tgt, fa in TARGETS_T9.items():
            pa = fa(tag)
            rows.append(dict(detector=LABEL[tag], source=src, target=tgt, **{f't_{k}': T[k] for k in KEYS},
                             **{f'FNR_{k}': fnr(pa, T[k]) for k in KEYS}))
T9 = pd.DataFrame(rows); T9.to_csv('reports/source_threshold_interval.csv', index=False)
piv = T9.pivot_table(index=['detector','target'], columns='source', values=['FNR_lo','FNR_hi','FNR_guar'])
print('=== Table 9 as intervals: FNR on each target, per source threshold ===')
print(piv.round(3).to_string())
print('\nOracle (bipia_host source) on the indirect targets, lo / hi / guar:')
print(T9[(T9.source=='bipia_host') & (T9.target.str.startswith('indirect'))][['detector','target','FNR_lo','FNR_hi','FNR_guar']].round(3).to_string(index=False))

## Stage E. Table 5 exploitability as intervals (needs phase1 caches)

Per-item downstream success is rebuilt exactly as notebook 07 did, from `data/phase1/out_{target}_atk.json` and `data/phase1/judge_{target}.json` through `src/attack_success.py`. Then the miss set is taken at each of the three thresholds. Skips with a message if the caches are absent.

In [ ]:
import attack_success as asuc
atk = bip[bip.label == 1].reset_index(drop=True)
cats = atk.meta.tolist(); instrs = atk.text.str.split('\n').str[-1].tolist()
target_tags = sorted(os.path.basename(f)[4:-9] for f in glob.glob('data/phase1/out_*_atk.json'))
EX = None
if not target_tags:
    print('phase1 caches not found (data/phase1/out_*_atk.json); skipping Table 5')
else:
    def success_vector(tt):
        outs = json.load(open(f'data/phase1/out_{tt}_atk.json'))
        jf = f'data/phase1/judge_{tt}.json'; jv = json.load(open(jf)) if os.path.exists(jf) else {}
        assert len(outs) == len(atk), f'{tt}: {len(outs)} outputs vs {len(atk)} attacks'
        return np.array([bool(asuc.combine(asuc.programmatic_success(cats[i], instrs[i], outs[i]), jv.get(str(i), False))) for i in range(len(atk))])
    SUCC = {tt: success_vector(tt) for tt in target_tags}
    atk_mask = bip.label.values == 1
    rows = []
    for tag in TAGS:
        pa = P[tag]['bipia_cat'][atk_mask]; T = three_thresholds(CELLS[tag][1])
        for tt, s in SUCC.items():
            r = dict(detector=LABEL[tag], target=tt, n_atk=int(len(s)), ASR_overall=float(s.mean()))
            for k in KEYS:
                miss = pa < T[k]
                r[f'n_miss_{k}'] = int(miss.sum())
                r[f'ASR_miss_{k}'] = float(s[miss].mean()) if miss.sum() else np.nan
                r[f'ASR_caught_{k}'] = float(s[~miss].mean()) if (~miss).sum() else np.nan
                r[f'leak_of_success_{k}'] = float(miss[s].mean()) if s.sum() else np.nan
                r[f'exploitable_miss_{k}'] = float((miss & s).mean())
            rows.append(r)
    EX = pd.DataFrame(rows); EX.to_csv('reports/exploitable_miss_interval.csv', index=False)
    print('=== Table 5 as intervals ===')
    print(EX.round(3).to_string(index=False))

## Stage F. Adversary evasion (Section 4.11) as intervals (needs phase5 caches)

Aggregate static and adaptive FNR at the three thresholds, from `data/phase5/score_{tag}.npy` with the variant grouping recovered from `data/phase5/rewrites.npy`. Only aggregates are computed, so the result does not depend on the order of the stratified subsample. The manufactured-miss rate needs the phase5 functionality judging and is not re-derived here.

In [ ]:
AD = None
rw_fp = 'data/phase5/rewrites.npy'
if not os.path.exists(rw_fp):
    print('phase5 caches not found; skipping adversary interval')
else:
    REW = np.load(rw_fp, allow_pickle=True)
    counts = np.array([1 + len(r) for r in REW])            # orig + rewrites per attack
    owner = np.repeat(np.arange(len(REW)), counts)
    first = np.r_[0, np.cumsum(counts)[:-1]]                  # index of each attack's original
    rows = []
    for tag in ['protectai_v2', 'prompt_guard_2']:
        fp = f'data/phase5/score_{tag}.npy'
        if not os.path.exists(fp): print('missing', fp); continue
        sc = np.load(fp); assert len(sc) == counts.sum(), f'{fp}: {len(sc)} scores vs {counts.sum()} variants'
        p_static = sc[first]
        p_adapt = np.array([sc[owner == i].min() for i in range(len(REW))])
        T = three_thresholds(CELLS[tag][1])
        r = dict(detector=LABEL[tag], n_atk=int(len(REW)), mean_p_static=float(p_static.mean()), mean_p_adapt=float(p_adapt.mean()))
        for k in KEYS:
            r[f'FNR_static_{k}'] = fnr(p_static, T[k]); r[f'FNR_adapt_{k}'] = fnr(p_adapt, T[k])
            r[f'dFNR_{k}'] = r[f'FNR_adapt_{k}'] - r[f'FNR_static_{k}']
        rows.append(r)
    AD = pd.DataFrame(rows); AD.to_csv('reports/adversary_interval.csv', index=False)
    print('=== Section 4.11 evasion as intervals ===')
    print(AD.round(3).to_string(index=False))

## Stage G. Table 7 judge panel as intervals (needs phase3 caches)

In [ ]:
JD = None
jfiles = {k: f'data/phase3/judge_{k}.npy' for k in FRAMES}
Pj = {}
for k, f in jfiles.items():
    if not os.path.exists(f): print('judge cache missing, shift skipped:', f); continue
    a = np.load(f)
    if len(a) != len(FRAMES[k]): print(f'judge_{k}.npy has {len(a)} scores vs {len(FRAMES[k])} rows; shift skipped'); continue
    Pj[k] = a
if 'deepset' not in Pj:
    print('judge deepset cache unavailable; Table 7 skipped (no calibration benigns)')
else:
    def finite(y, p):
        m = np.isfinite(p); return y[m], p[m]
    ben_cal = Pj['deepset'][yd == 0]; ben_cal = ben_cal[np.isfinite(ben_cal)]
    jc = {'direct': finite(yd, Pj['deepset'])}
    if 'bipia_cat' in Pj:
        jc['indirect_harmful'] = finite(np.r_[np.ones(H.sum()), np.zeros(bipben.sum())], np.r_[Pj['bipia_cat'][H], Pj['bipia_cat'][bipben]])
        jc['indirect_hijack']  = finite(np.r_[np.ones(J.sum()), np.zeros(bipben.sum())], np.r_[Pj['bipia_cat'][J], Pj['bipia_cat'][bipben]])
    if 'jailbreak' in Pj:
        jc['jailbreak'] = finite(jb.label.values, Pj['jailbreak'])
    rows = []
    for sh, (y, p) in jc.items():
        pa, pb = p[y == 1], p[y == 0]
        r = interval_cell(pa, ben_cal, pb_tgt=pb, auroc_fn=auroc)
        rows.append(dict(detector='LLM-judge (Qwen2.5-7B)', shift=sh, n=int(len(p)), n_atk=int(len(pa)), **r))
    JD = pd.DataFrame(rows); JD.to_csv('reports/judge_panel_interval.csv', index=False)
    print('=== Table 7 as intervals ===')
    print(JD[['detector','shift','n','n_atk','t_lo','t_hi','t_guar','FNR_lo','FNR_hi','FNR_guar','S_lo','S_hi','S_guar','tgtFPR_guar','AUROC']].round(4).to_string(index=False))

## Stage H. Figure: the FNR interval per cell

In [ ]:
import matplotlib.pyplot as plt
d = PI[PI['shift'].isin(SHIFTS)].copy()
assert len(d) == len(TAGS) * len(SHIFTS), f'figure selector returned {len(d)} rows'
fig, ax = plt.subplots(figsize=(11, 4.8))
labels = [f"{r.detector}\n{r['shift']}" for _, r in d.iterrows()]
x = np.arange(len(d))
ax.vlines(x, d.FNR_lo, d.FNR_hi, color='C0', lw=6, alpha=.5, label='identified interval (lo to hi, same calibration FPR)')
ax.scatter(x, d.FNR_guar, color='C3', marker='_', s=300, lw=2, label='guaranteed FPR <= 1 percent')
pub = {(LABEL['protectai_v2'],'direct'):0.548,(LABEL['protectai_v2'],'indirect_harmful'):0.650,(LABEL['protectai_v2'],'indirect_hijack'):0.693,(LABEL['protectai_v2'],'jailbreak'):0.136,
       (LABEL['prompt_guard_2'],'direct'):0.532,(LABEL['prompt_guard_2'],'indirect_harmful'):0.217,(LABEL['prompt_guard_2'],'indirect_hijack'):0.693,(LABEL['prompt_guard_2'],'jailbreak'):0.010,
       (LABEL['prompt_guard_2_22m'],'direct'):0.837,(LABEL['prompt_guard_2_22m'],'indirect_harmful'):0.900,(LABEL['prompt_guard_2_22m'],'indirect_hijack'):0.967,(LABEL['prompt_guard_2_22m'],'jailbreak'):0.083}
ax.scatter(x, [pub[(r.detector, r['shift'])] for _, r in d.iterrows()], color='k', marker='o', s=25, zorder=3, label='published (linear interpolation)')
ax.set_xticks(x); ax.set_xticklabels(labels, rotation=60, ha='right', fontsize=7); ax.set_ylim(0, 1.02)
ax.set_ylabel('false-negative rate'); ax.set_title('FNR is identified only up to an interval at a 1 percent FPR operating point on 399 benigns')
ax.grid(axis='y', alpha=.3); ax.legend(fontsize=8, loc='lower right')
plt.tight_layout(); plt.savefig('figures/panel_interval.png', dpi=150); plt.show()

## Stage I. Log and commit

In [ ]:
from reslog import log_result
parts = ['Table 2 intervals:\n' + PI[show].round(3).to_string(index=False),
         'Table 4 CCI intervals:\n' + CCI.round(3).to_string(index=False)]
if EX is not None: parts.append('Table 5 intervals:\n' + EX.round(3).to_string(index=False))
if AD is not None: parts.append('Adversary intervals:\n' + AD.round(3).to_string(index=False))
if JD is not None: parts.append('Judge intervals:\n' + JD[['shift','FNR_lo','FNR_hi','FNR_guar','AUROC']].round(3).to_string(index=False))
summary = '\n\n'.join(parts)
log_result('nb15: interval re-report of all threshold-dependent tables', summary, csv_df=PI, csv_name='panel_interval.csv')
print(summary)

# === STANDARD COMMIT CELL ===
import os, subprocess
os.chdir(REPO_DIR)
r = subprocess.run(["python", "tools/commit_cell.py",
                    "nb15: interval re-report of Tables 2, 3, 4, 5, 7, 9 and the adversary; add src/interval_panel.py; commit tool gains a secret scan"],
                   capture_output=True, text=True)
print(r.stdout); print(r.stderr)